In [3]:
import sys
from pathlib import Path

# Support running notebook from either project root or notebooks/.
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for base in (cwd, cwd.parent):
    if (base / "models").exists():
        PROJECT_ROOT = base
        if str(base) not in sys.path:
            sys.path.insert(0, str(base))
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Could not locate project root (folder containing 'models').")

In [4]:
import torch

from generate import Generator_Qwen_3
from models import Qwen_3_Model, get_qwen3_config
from utils import get_device


In [5]:
config = get_qwen3_config("0.6B")

model = Qwen_3_Model(**config)

device = get_device()
model.to(device)

checkpoint_path = PROJECT_ROOT / "checkpoints" / "qwen3_0.6b_base.pth"
if not checkpoint_path.exists():
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

state_dict = torch.load(checkpoint_path, map_location="mps")
if isinstance(state_dict, dict) and "model_state_dict" in state_dict:
    state_dict = state_dict["model_state_dict"]

model.load_state_dict(state_dict)

Using MPS device


<All keys matched successfully>

In [6]:
generator = Generator_Qwen_3(
    model=model,
    num_layers=config["num_layers"],
    tokenizer_file_path=PROJECT_ROOT / "tokenizer" / "qwen_3_instruct_tokenizer.json",
    model_type="base",
    model_size="0.6B",
    apply_chat_template=False,
    add_generation_prompt=False,
    add_thinking=False,
)

In [7]:
output = generator.generate(
    idx=generator.text_to_token_ids(
        text="You are a helpful assistant, who follows the instruction given next. Explain LLMs in 20 words?"
    ).unsqueeze(0),
    max_token_length=150,
    # temperature=0.7,
    # top_k=50,
    cache_enabled=True,
)

print(generator.token_ids_to_text(output.squeeze(0)))

You are a helpful assistant, who follows the instruction given next. Explain LLMs in 20 words? Large Language Models are AI systems that can understand and generate human language.<|endoftext|>
